## Example 1: AttrDict and FrozenAttrDict

In [ ]:
# Example 1: AttrDict for enhanced dictionary access
from monty.collections import AttrDict, FrozenAttrDict

print("Example 1 - AttrDict and FrozenAttrDict")

# Create AttrDict with attribute access
config = AttrDict()
config.name = "experiment"
config.learning_rate = 0.001
config.batch_size = 32

print(f"AttrDict: {config.name}, lr={config.learning_rate}")

# Dictionary access still works
config['epochs'] = 100
print(f"Mixed access: {config.epochs}")

# Create from existing dict
data = {'model': 'neural_net', 'layers': [128, 64, 32]}
model_config = AttrDict(data)
print(f"From dict: {model_config.model}")

# FrozenAttrDict for immutability
frozen = FrozenAttrDict(config)
print(f"Frozen: {frozen.name}")

try:
    frozen.name = "modified"
except TypeError as e:
    # FrozenAttrDict refuses attribute assignment with a TypeError; nothing else is expected here.
    assert "does not support item assignment" in str(e), e
    print(f"Immutable: {type(e).__name__}")
else:
    raise AssertionError("FrozenAttrDict accepted an attribute assignment it must refuse")
assert frozen.name == "experiment", frozen.name

print("Example 1 completed")

## Example 2: MSONable JSON serialization

In [ ]:
# Example 2: JSON serialization with MSONable
from monty.json import MSONable, MontyEncoder, MontyDecoder
import json

print("Example 2 - MSONable JSON serialization")

class Model(MSONable):
    def __init__(self, name, params):
        self.name = name
        self.params = params
    
    def as_dict(self):
        return {"@module": self.__class__.__module__, "@class": self.__class__.__name__,
                "name": self.name, "params": self.params}
    
    @classmethod
    def from_dict(cls, d):
        return cls(d["name"], d["params"])

# Create and serialize
model = Model("neural_net", {"layers": [128, 64], "lr": 0.001})
json_str = json.dumps(model, cls=MontyEncoder)
print(f"Serialized: {len(json_str)} chars")

# Deserialize
restored = json.loads(json_str, cls=MontyDecoder)
print(f"Restored: {restored.name}, params={restored.params}")
print(f"Type preserved: {type(restored).__name__}")
assert json.loads(json_str)["@class"] == "Model"
assert type(restored) is Model
assert restored.name == "neural_net" and restored.params == {"layers": [128, 64], "lr": 0.001}

print("Example 2 completed")

## Example 3: Design patterns and string utilities

In [ ]:
# Example 3: Design patterns and utilities
from monty.design_patterns import singleton, cached_class
from monty.functools import lazy_property
from monty.string import remove_non_ascii

print("Example 3 - Design patterns and utilities")

@singleton
class Config:
    def __init__(self):
        self.value = 42

c1 = Config()
c2 = Config()
print(f"Singleton: {c1 is c2}")
assert c1 is c2 and c1.value == 42

@cached_class
class Processor:
    def __init__(self, name):
        self.name = name

p1 = Processor("test")
p2 = Processor("test")
print(f"Cached: {p1 is p2}")
# cached_class returns the same instance for the same constructor arguments only
assert p1 is p2 and Processor("other") is not p1

# String utilities
text = "Café™ résumé"
clean = remove_non_ascii(text)
print(f"Original: {text}")
print(f"Cleaned: {clean}")
assert clean == "Caf rsum", clean

print("Example 3 completed")

## Example 4: Temporary directories and file search

In [ ]:
# Example 4: System utilities
from monty.os.path import find_exts, zpath
from monty.tempfile import ScratchDir
import os

print("Example 4 - System utilities")

# Temporary directory with auto-cleanup
with ScratchDir(".") as scratch:
    print(f"Scratch dir: {os.path.basename(scratch)}")
    
    # Create test files
    for ext in ['txt', 'json', 'csv']:
        file_path = os.path.join(scratch, f"test.{ext}")
        with open(file_path, 'w') as f:
            f.write(f"test content")
    
    # Find files by extension
    txt_files = find_exts(scratch, "txt")
    json_files = find_exts(scratch, "json")
    print(f"TXT files: {len(txt_files)}")
    print(f"JSON files: {len(json_files)}")
    assert [os.path.basename(p) for p in txt_files] == ["test.txt"], txt_files
    assert [os.path.basename(p) for p in json_files] == ["test.json"], json_files

assert not os.path.exists(scratch), scratch
print("Scratch dir cleaned up")
print("Example 4 completed")

## Example 5: Compressed I/O and JSON sanitization

In [ ]:
# Example 5: Comprehensive integration
from monty.collections import AttrDict
from monty.json import MSONable, jsanitize
from monty.io import zopen
import os
import tempfile
import json

print("Example 5 - Comprehensive integration")

# Configuration with nested AttrDict
config = AttrDict()
config.model = AttrDict({'name': 'neural_net', 'layers': [128, 64]})
config.training = AttrDict({'epochs': 100, 'lr': 0.001})
print(f"Config: {config.model.name}, {config.training.epochs} epochs")

# Compressed I/O
with tempfile.NamedTemporaryFile(suffix='.json.gz', delete=False) as tmp:
    temp_path = tmp.name
    with zopen(temp_path, 'wt') as f:
        json.dump({'data': list(range(100))}, f)
    
    with zopen(temp_path, 'rt') as f:
        loaded = json.load(f)
    
    print(f"Compressed I/O: {len(loaded['data'])} items")
    assert loaded == {'data': list(range(100))}
    # zopen picked gzip from the .gz suffix
    with open(temp_path, 'rb') as raw:
        assert raw.read(2) == b'\x1f\x8b'
os.remove(temp_path)

# JSON sanitization
sanitized = jsanitize(config)
print(f"Sanitized: {type(sanitized).__name__}")
assert sanitized == {'model': {'name': 'neural_net', 'layers': [128, 64]}, 'training': {'epochs': 100, 'lr': 0.001}}
assert json.loads(json.dumps(sanitized)) == sanitized

print("Example 5 completed")